# Synthetic Ramesside Star Clocks

In [1]:
# set up paths to local module
import sys 
from pathlib import Path
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))

import decanopy


### Import Sky & Initialize 

In [2]:
from decanopy.skyproc.sky_processing import initialize_sky

### Choose sky data for synRSC calculations
# -----------------------------------------------------------------------------
# Two current options are listed below; 
# the "hippdata1300BC.txt" and "mockdata_518_1300BC-Mar-18-2024_1059.txt" don't ship with GitHub,
# so they have to be manually added to the correct folders! 
# Will also need star_data_Mar-18-2024_1059.csv for OPTION 2. It must exist in /data/input/skyflow/rand_sky/.

# For now, uncomment one of the two options below to choose sky data.
# -----------------------------------------------------------------------------

## OPTION 1: Real sky data from Hipparcos catalog
skytype = "real_sky" # options: {"real_sky", "rand_sky", "star_like"}
skyname = "hippdata1300BC.txt" # must exist in /data/output/skyflow/<skytype>


## OPTION 2: Random sky data from isotropic sky distribution
# skytype = "rand_sky" # options: {"real_sky", "rand_sky", "star_like"}
# skyname = "mockdata_518_1300BC-Mar-18-2024_1059.txt" # must exist in /data/output/skyflow/<skytype>


skydict = initialize_sky(skyname, skytype) # takes about 1 min for ~500 stars 

### Write to Excel

In [ ]:
# INPUT: name save folder

dirname = "np_az_alt"  

In [ ]:
# make save folder inside output/skyflow/<skytype> 
import os
savepath = skydict['writepath'] / dirname
if not os.path.exists(savepath):
    os.makedirs(savepath)

In [4]:
# define bin and gap sizes
bsize = 1 # bin size (must be 1 if gsize = 0)
gsize = 0 # gap size (relative to binsize) 

# define altitude range
alt_min = 5
alt_max_start = 25
alt_max_end = 90

# example generated = (5, 25), (5, 26), (5, 27)...,(5,90)

# define horizon range 
hor_head = 180 # degrees, center of horizon
hor_min = 4 # degrees of width centered on hor_head 
hor_max = 20 # degrees of width centered on hor_head

# example generated (178, 182), (177.5, 182.5), ... (170, 190)

In [6]:
# run the loop!
from decanopy.models.RSC.syn_rsc import write_synRSC_to_excel
count = 0

# iterate horizon widths
for hor_width in range(hor_min, hor_max +1, 1):
    horizon = (hor_head - hor_width/2, hor_head + hor_width/2)
    # iterate max altitude 
    for alt_max in range(alt_max_start, alt_max_end+1, 1): # iterate. by 1 degree
        alt_window = (alt_min, alt_max) # define altitude 
        writename = f"results_h{horizon[0]}-{horizon[1]}_a{alt_min}-{alt_max}.xlsx" # define name of file 
        #print(writename)
        count += 1
        #writename = 'alt_' + str(alt_window).replace(" ", "") + '.xlsx' 
        try:
            write_synRSC_to_excel(writename, horizon, alt_window, bsize, gsize, skydict, clobberSave=False, writepath=savepath)
        except FileExistsError as e:
            print(f"Skipping: {e}")
            continue  # skips to next iteration of inner loop

print(f"Loop generated {count} parameter combinations.") 

# NOTE: clobberSave=True will overwrite existing files without asking! 
# However, it will warn you that a file was overwritten and point to it. 
# clobberSave = False by default, which will raise an error if the file exists.

Loop generated 1122 parameter combinations.


### Generate summary dictionary

In [ ]:
# generate above dictionary with summary of all data in savepath 

import pickle
import re
from decanopy.models.RSC import heat_mapping_helper as hm

df = hm.scan_folder(savepath, cell_re=hm._HCELL_RE)

In [ ]:
# Name dictionary same as save folder with "_dict" appended to it
# (keeping this structure will make importing easier later)

dictname = dirname + "_dict" 

# Save dictionary
with open(savepath / dict_name, "wb") as f:
    pickle.dump(df, f)

# print success message
print(f"Dictionary saved at {savepath / dict_name}.") 

Dictionary saved at /Users/lunazagor/Code/GitHub/decanOpy/data/output/rsc/real_sky/np_az_alt/np_az_alt_dict.
